In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from experiments import setup_publication_style

RESULTS_DIR = Path("../../results/paper/hyperparameters/aif")
OUTPUT_DIR = Path("../../results/paper/hyperparameters/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARAM_NAMES = [
    "pB_lr", "c_preference", "noisy_A",
    "policy_len", "update_interval",
    "pB_scale", "gamma", "alpha", "bias",
]

PARAM_LABELS = {
    "pB_lr": "pB Learning Rate",
    "c_preference": "Cooperative Preference",
    "noisy_A": "Noisy Observation Model",
    "policy_len": "Policy Length",
    "update_interval": "Update Interval",
    "pB_scale": "pB Scale",
    "gamma": "Gamma",
    "alpha": "Alpha",
    "bias": "Bias",
}

setup_publication_style(use_latex=False, font_size=10)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

print(f"Results directory: {RESULTS_DIR.resolve()}")
print(f"Output directory:  {OUTPUT_DIR.resolve()}")

In [ ]:
# ── Opponent Configuration ────────────────────────────────────────────────────
# Choose which matches to analyze, always from the AIF agent's perspective
# (Player index == 0).
#
#   "self"  →  AIF vs AIF          (Opponent index == 0)
#   "tft"   →  AIF vs Tit For Tat  (Opponent index == 1)
#   "all"   →  all opponents combined  (no additional filter)

OPPONENT = "tft"

_OPPONENT_CONFIGS = {
    "self": {"opponent_index": 0, "label": "AIF Self-play"},
    "tft":  {"opponent_index": 1, "label": "vs Tit For Tat"},
    "all":  {"opponent_index": None, "label": "All Opponents"},
}

assert OPPONENT in _OPPONENT_CONFIGS, f"OPPONENT must be one of {list(_OPPONENT_CONFIGS)}"

OPPONENT_INDEX = _OPPONENT_CONFIGS[OPPONENT]["opponent_index"]
OPPONENT_LABEL = _OPPONENT_CONFIGS[OPPONENT]["label"]

# ── Aggregation Configuration ─────────────────────────────────────────────────
# For each line (one per value of the varied parameter), how to handle the
# other hyperparameter combinations:
#
#   "average"  →  mean ± 95 % CI across all other combinations
#   "best"     →  single combination with the highest mean score averaged
#                 over all noise levels (no CI band; best combo shown in label)

AGGREGATION = "average"

assert AGGREGATION in ("average", "best"), "AGGREGATION must be 'average' or 'best'"

print(f"Opponent filter : {OPPONENT_LABEL!r}  (opponent_index={OPPONENT_INDEX})")
print(f"Aggregation     : {AGGREGATION!r}")

In [ ]:
def parse_dir_name(dir_name: str) -> dict:
    """Parse AIF_<pB_lr>_<c_pref>_<noisy_A>_<policy_len>_<update_interval>_<pB_scale>_<gamma>_<alpha>_<bias>."""
    parts = dir_name.split("_")
    return {
        "pB_lr":           float(parts[1]),
        "c_preference":    parts[2],
        "noisy_A":         parts[3],
        "policy_len":      int(parts[4]),
        "update_interval": int(parts[5]),
        "pB_scale":        float(parts[6]),
        "gamma":           float(parts[7]),
        "alpha":           float(parts[8]),
        "bias":            float(parts[9]),
    }


def load_aif_metrics(csv_file: Path, opponent_idx: int | None) -> dict:
    """Return score and mutual-cooperation rate for the AIF agent (Player index == 0),
    optionally restricted to matches against a specific opponent index.

    CC rate = total CC count / total turns across all matching rows.
    Returns {"score": float, "cc_rate": float}, NaN if no rows match.
    """
    data = pd.read_csv(csv_file)
    mask = data["Player index"] == 0
    if opponent_idx is not None:
        mask &= data["Opponent index"] == opponent_idx
    rows = data.loc[mask]
    if len(rows) == 0:
        return {"score": float("nan"), "cc_rate": float("nan")}
    score = float(rows["Score"].mean())
    total_turns = rows["Turns"].sum()
    cc_rate = float(rows["CC count"].sum() / total_turns) if total_turns > 0 else float("nan")
    return {"score": score, "cc_rate": cc_rate}


rows = []

for config_dir in sorted(RESULTS_DIR.iterdir()):
    if not config_dir.is_dir() or config_dir.name.startswith("."):
        continue

    params = parse_dir_name(config_dir.name)
    rep_dirs = sorted(config_dir.glob("repetition_*"))
    if not rep_dirs:
        continue

    noise_levels = sorted(float(f.stem) for f in rep_dirs[0].glob("*.csv"))

    for noise in noise_levels:
        rep_scores, rep_cc_rates = [], []
        for rep_dir in rep_dirs:
            csv_file = rep_dir / f"{noise}.csv"
            if csv_file.exists():
                m = load_aif_metrics(csv_file, OPPONENT_INDEX)
                if np.isfinite(m["score"]):
                    rep_scores.append(m["score"])
                if np.isfinite(m["cc_rate"]):
                    rep_cc_rates.append(m["cc_rate"])

        if rep_scores:
            rows.append({
                **params,
                "noise_level":   noise,
                "mean_score":    float(np.mean(rep_scores)),
                "rep_scores":    rep_scores,    # per-rep scores for CI in "best" mode
                "mean_cc_rate":  float(np.mean(rep_cc_rates)) if rep_cc_rates else float("nan"),
                "rep_cc_rates":  rep_cc_rates,  # per-rep CC rates for CI in "best" mode
            })

df = pd.DataFrame(rows)
n_configs = df[PARAM_NAMES].drop_duplicates().shape[0]
print(f"Loaded {len(df)} rows — {n_configs} configs × {df['noise_level'].nunique()} noise levels")
print(f"Opponent filter: {OPPONENT_LABEL!r}")
df.head()

In [ ]:
varying_params = [p for p in PARAM_NAMES if df[p].nunique() > 1]
print(f"Parameters with >1 unique value in the data: {varying_params}")

for p in varying_params:
    print(f"  {PARAM_LABELS[p]}: {sorted(df[p].unique())}")

In [ ]:
def t_ci95(arr: np.ndarray) -> tuple[float, float, float]:
    """Return (mean, ci_lower, ci_upper) using a 95% t-interval."""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan
    mean = float(np.mean(arr))
    if n == 1:
        return mean, mean, mean
    std = float(np.std(arr, ddof=1))
    if std <= 0 or not np.isfinite(std):
        return mean, mean, mean
    lo, hi = stats.t.interval(0.95, n - 1, loc=mean, scale=std / np.sqrt(n))
    return mean, float(lo), float(hi)


def best_config_for(subset: pd.DataFrame) -> dict:
    """Return the param values of the config with the highest mean score
    averaged across all noise levels."""
    config_avg = subset.groupby(PARAM_NAMES)["mean_score"].mean()
    best_tuple = config_avg.idxmax()
    return dict(zip(PARAM_NAMES, best_tuple))


def plot_metric_lines(ax, subset, metric_col, rep_col, param, param_label,
                      noise_levels, val, c, m, label):
    """Draw one line (+ optional CI band) for a single param value on ax."""
    if AGGREGATION == "best":
        best = best_config_for(subset)
        mask = pd.Series(True, index=subset.index)
        for col, bval in best.items():
            mask &= subset[col] == bval
        best_rows = subset[mask].sort_values("noise_level")
        ax.plot(
            best_rows["noise_level"], best_rows[metric_col],
            marker=m, color=c, label=label, linewidth=1.5, markersize=4,
        )
    else:  # "average"
        noise_data = []
        for noise in noise_levels:
            vals = subset.loc[subset["noise_level"] == noise, metric_col].values
            mean, lo, hi = t_ci95(vals)
            noise_data.append({"noise": noise, "mean": mean, "lo": lo, "hi": hi})
        nd = pd.DataFrame(noise_data)
        ax.plot(nd["noise"], nd["mean"], marker=m, color=c,
                label=label, linewidth=1.5, markersize=4)
        ax.fill_between(nd["noise"], nd["lo"], nd["hi"], alpha=0.15, color=c)


colors = plt.cm.tab10.colors
markers = ["o", "s", "^", "D", "v", "<", ">", "p", "*", "h"]

for param in varying_params:
    param_label = PARAM_LABELS[param]
    param_values = sorted(df[param].unique())
    noise_levels = sorted(df["noise_level"].unique())
    agg_label = "mean ± 95% CI" if AGGREGATION == "average" else "best combo"

    fig, (ax_score, ax_cc) = plt.subplots(2, 1, figsize=(5, 6), sharex=True)

    for idx, val in enumerate(param_values):
        subset = df[df[param] == val]
        c = colors[idx % len(colors)]
        m = markers[idx % len(markers)]

        # Build legend label (shared across both panels)
        if AGGREGATION == "best":
            best = best_config_for(subset)
            other_best = {PARAM_LABELS[p]: best[p] for p in varying_params if p != param}
            if other_best:
                combo_str = ", ".join(f"{k}={v}" for k, v in other_best.items())
                label = f"{param_label}={val}  [{combo_str}]"
            else:
                label = f"{param_label}={val}"
        else:
            label = f"{param_label} = {val}"

        plot_metric_lines(ax_score, subset, "mean_score",  "rep_scores",
                          param, param_label, noise_levels, val, c, m, label)
        plot_metric_lines(ax_cc,    subset, "mean_cc_rate", "rep_cc_rates",
                          param, param_label, noise_levels, val, c, m, label)

    ax_score.set_ylabel("Mean Payoff")
    ax_score.set_title(
        f"Mean Payoff vs Noise Level — {OPPONENT_LABEL}\n(by {param_label}, {agg_label})"
    )
    ax_score.grid(True, linestyle="--", alpha=0.3)
    ax_score.legend(loc="best", frameon=True, framealpha=0.9, edgecolor="gray")

    ax_cc.set_xlabel(r"Noise Level ($\epsilon$)")
    ax_cc.set_ylabel("Mutual Cooperation Rate (CC / turns)")
    ax_cc.set_ylim(-0.05, 1.05)
    ax_cc.grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()

    stem = f"hyperparams_{param}_{OPPONENT}_{AGGREGATION}"
    for fmt in ("pdf", "png"):
        plt.savefig(OUTPUT_DIR / f"{stem}.{fmt}", dpi=600, bbox_inches="tight")

    plt.show()
    print(f"Saved: {stem}.pdf / .png")